In [ ]:
import pandas as pd
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi
import isodate
import os
import requests

# =========================
# CONFIG
# =========================
API_KEY = "A***********************s"
INPUT_CSV = "videos.csv"
OUTPUT_CSV = "youtube_videos.csv"
THUMBNAIL_DIR = "thumbnails_clickbait"

os.makedirs(THUMBNAIL_DIR, exist_ok=True)

youtube = build("youtube", "v3", developerKey=API_KEY)

# =========================
# LOAD CSV (FIXED ENCODING)
# =========================
df = pd.read_csv(INPUT_CSV, encoding="latin1")
video_ids = df["video_id"].astype(str).tolist()

# =========================
# GET CATEGORY MAPPING
# =========================
def get_video_categories(region="US"):
    request = youtube.videoCategories().list(
        part="snippet",
        regionCode=region
    )
    response = request.execute()

    categories = {}
    for item in response["items"]:
        categories[item["id"]] = item["snippet"]["title"]
    return categories

category_map = get_video_categories()

# =========================
# FUNCTIONS
# =========================
def fetch_video_details(video_id):
    results = {}

    for i in range(0, len(video_ids), 50):
        batch = video_id[i:i+50]

        request = youtube.videos().list(
            part="snippet,statistics,contentDetails,topicDetails",
            id=",".join(batch)
        )
        response = request.execute()

        for item in response.get("items", []):
            vid = item["id"]
            snippet = item.get("snippet", {})
            stats = item.get("statistics", {})
            content = item.get("contentDetails", {})
            topic = item.get("topicDetails", {})

            # Channel info
            channel_id = snippet.get("channelId", "")
            channel_title = snippet.get("channelTitle", "")

            # Description (NEW)
            description = snippet.get("description", "")

            # Duration
            duration_iso = content.get("duration", "PT0S")
            duration_seconds = int(isodate.parse_duration(duration_iso).total_seconds())

            # Stats
            view_count = int(stats.get("viewCount", 0))
            like_count = int(stats.get("likeCount", 0))
            comment_count = int(stats.get("commentCount", 0))

            likes_per_view = like_count / view_count if view_count > 0 else 0
            comments_per_view = comment_count / view_count if view_count > 0 else 0

            # Topics
            topic_ids = topic.get("topicIds", [])
            topic_text = ", ".join(topic_ids)

            # Category
            category_id = snippet.get("categoryId", "")
            category_name = category_map.get(category_id, "")

            # Tags
            tags = snippet.get("tags", [])
            tags_text = ", ".join(tags)

            # Thumbnail
            thumbnails = snippet.get("thumbnails", {})
            thumb_url = thumbnails.get("high", thumbnails.get("default", {})).get("url", "")

            results[vid] = {
                "duration_seconds": duration_seconds,
                "view_count": view_count,
                "like_count": like_count,
                "comment_count": comment_count,
                "likes_per_view": likes_per_view,
                "comments_per_view": comments_per_view,
                "topic_details": topic_text,
                "video_category": category_name,
                "tags": tags_text,
                "thumbnail_url": thumb_url,
                "channel_id": channel_id,
                "channel_title": channel_title,
                "description": description   # ✅ NEW
            }

    return results

def get_video_title(video_id):
    try:
        request = youtube.videos().list(
            part="snippet",
            id=video_id
        )
        response = request.execute()
        
        if response["items"]:
            return response["items"][0]["snippet"]["title"]
        else:
            return "Not Found"
    except Exception as e:
        return "Error"
    
df["video_title"] = df["video_id"].apply(get_video_title)

    
def fetch_channel_subscribers(channel_ids):
    results = {}

    for i in range(0, len(channel_ids), 50):
        batch = channel_ids[i:i+50]

        request = youtube.channels().list(
            part="statistics",
            id=",".join(batch)
        )
        response = request.execute()

        for item in response.get("items", []):
            cid = item["id"]
            stats = item.get("statistics", {})
            subs = int(stats.get("subscriberCount", 0))
            results[cid] = subs

    return results


def download_thumbnail(video_id, url):
    if url == "":
        return ""

    path = os.path.join(THUMBNAIL_DIR, f"{video_id}.jpg")
    try:
        r = requests.get(url, timeout=10)
        with open(path, "wb") as f:
            f.write(r.content)
        return path
    except:
        return ""


def fetch_top_comments(video_id, max_comments=5):
    comments = []
    try:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=max_comments,
            order="relevance",
            textFormat="plainText"
        )
        response = request.execute()

        for item in response.get("items", []):
            text = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
            comments.append(text)
    except:
        comments = []

    return " ||| ".join(comments)



# =========================
# FETCH VIDEO METADATA
# =========================
video_info = fetch_video_details(video_ids)

channel_ids = list(set([
    info.get("channel_id", "") for info in video_info.values()
]))

channel_subscriber_map = fetch_channel_subscribers(channel_ids)

# =========================
# ADD NEW COLUMNS
# =========================
video_title_list=[]
view_count_list = []
like_count_list = []
comment_count_list = []
duration_list = []
likes_per_view_list = []
comments_per_view_list = []
category_list = []
tags_list = []
thumbnail_url_list = []
thumbnail_path_list = []
top_comments_list = []
channel_title_list = []
subscriber_list = []
description_list = []  

for vid in df["video_id"]:
    vid = str(vid)
    info = video_info.get(vid, {})

    view_count_list.append(info.get("view_count", 0))
    like_count_list.append(info.get("like_count", 0))
    comment_count_list.append(info.get("comment_count", 0))
    duration_list.append(info.get("duration_seconds", 0))

    likes_per_view_list.append(info.get("likes_per_view", 0))
    comments_per_view_list.append(info.get("comments_per_view", 0))
    category_list.append(info.get("video_category", ""))
    tags_list.append(info.get("tags", ""))

    thumb_url = info.get("thumbnail_url", "")
    thumbnail_url_list.append(thumb_url)
    thumbnail_path_list.append(download_thumbnail(vid, thumb_url))

    channel_title_list.append(info.get("channel_title", ""))
    channel_id = info.get("channel_id", "")
    subscriber_list.append(channel_subscriber_map.get(channel_id, 0))

    top_comments_list.append(fetch_top_comments(vid, 5))
    description_list.append(info.get("description", "")) 


# =========================
# SAVE TO DATAFRAME
# =========================
df['title'] = video_title_list
df["description"] = description_list 
df["thumbnail_url"] = thumbnail_url_list
df["thumbnail_path"] = thumbnail_path_list
df["view_count"] = view_count_list
df["like_count"] = like_count_list
df["comment_count"] = comment_count_list
df["duration_seconds"] = duration_list
df["likes_per_view"] = likes_per_view_list
df["comments_per_view"] = comments_per_view_list
df["channel_title"] = channel_title_list
df["channel_subscribers"] = subscriber_list
df["video_category"] = category_list
df["tags"] = tags_list
df["top_5_comments"] = top_comments_list
 

# =========================
# SAVE CSV (UTF-8)
# =========================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("✅ Enriched CSV saved as:", OUTPUT_CSV)

✅ Enriched CSV saved as: updated_file_features.csv
